# 협동로봇 관련 프로젝트

## 1. OpenCV기반 협동로봇 활용 이미지 제작 자동화 시스템

### 👉 역할
공정 물리 환경 제작, Input 이미지 데이터 제작, 로봇 팔 제어, GUI 제작

### 👉 개발 환경
하드웨어 - 협동로봇(Doosan m0609), PC, ipad

소프트웨어 - Ubuntu 22.04(ROS2 Humble), Python, Doosan API, OpenCV, RViz2, Procreate

### 👉 프로젝트 개요
Input 이미지를 OpenCV로 처리해 경로 좌표를 순차적으로 추출, 이를 로봇 팔 좌표로 변환해 자동으로 그림을 그리도록 구현.

### 👉 프로젝트 목표
실제 캔버스라는 공정 도구 특성에 맞춘 안정적인 드로잉 동작을 위한 힘 제어 설계 및 튜닝.

그리퍼가 그리기 도구(마커, 붓펜)을 안정적으로 파지하고 정확하게 원위치 시키게 하는 높은 동작 
안정성.

완성된 그림 품질을 위한 로봇팔의 선속도/각속도 최적화.

### 👉 주요성과 및 문제해결
원본 이미지 사용시 알고리즘 복잡도를 고려해 연결된 하나의 선으로 이루어진 이미지로 input 데이터 레이어 분할.

픽셀 단위 좌표 순차적 추출에 한계 느끼고 skeletonize + BFS기반 전처리 알고리즘을 적용해 추출.

힘 제어, 컴플라이언스 제어를 적용해 그림 도구와 캔버스 사이의 마찰력을 조절해 안정적인 드로잉 구현.

### Task 환경 구성
<img src="images/task.png" width="600">


### Image Data Layer 및 경로 추출 과정
<img src="images/image_layer.png" width="1000">


### RViz2 활용 시뮬레이션
<img src="images/simulation.png" width="1000">


### OpenCV 좌표 로봇 이동 좌표 변환 후 이동 제어 코드
설명: 이미지 → 스켈레톤 경로 추출 → 로봇 좌표 변환 → 협동로봇 드로잉 수행 + RViz 시각화

### 주요기능
1. 이미지 전처리 및 skeleton 추출
2. BFS 기반 경로 탐색
3. RViz 경로 시각화
4. 협동로봇 드로잉 제어
5. 힘제어 기반 인터랙션(Nudge)

### 사용 소스
- ROS2
- Doosan Robot API
- OpenCV
- Skeletonization
- BFS Path Planning
- RViz Visualization
- Compliance Control
- Force Control

In [ ]:
# ROS2 & Doosan
import rclpy, DR_init

# Python 기본 라이브러리
import time, math
from collections import deque

# 이미지 처리
import numpy as np
import cv2

# Skeleton image processing
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu
from skimage.morphology import skeletonize

# ROS visualization
from visualization_msgs.msg import Marker
from geometry_msgs.msg import Point

# Debug visualization
import matplotlib.pyplot as plt

ROBOT_ID = "dsr01"
ROBOT_MODEL = "m0609"
VELOCITY, ACC = 200, 200
DR_init.__dsr__id = ROBOT_ID
DR_init.__dsr__model = ROBOT_MODEL
OFF, ON = 0, 1

right_eyes_x = 15
right_eyes_y = -172

left_eyes_x = -40
left_eyes_y = -175

nose_offset_x = 20
nose_offset_y = 13

def main(args=None): # node 초기화 및 전체 드로잉 시퀀스 수행
    rclpy.init(args=args)
    node = rclpy.create_node("doosan_pick_and_place", namespace=ROBOT_ID)
    DR_init.__dsr__node = node

    try:
        from DSR_ROBOT2 import (
            set_tool, set_tcp, movej, movel, wait,
            set_digital_output, set_singularity_handling,
            movesx, DR_BASE, DR_MV_MOD_ABS, DR_MV_MOD_REL,task_compliance_ctrl,set_desired_force,DR_FC_MOD_REL,move_spiral,DR_AXIS_Z,DR_TOOL,release_force,
            release_compliance_ctrl,move_periodic,check_force_condition
        )
        from DR_common2 import posj, posx
    except ImportError as e:
        print(f"Error importing Doosan API modules: {e}")
        return

    set_tool("Tool Weight_test")
    set_tcp("test_TCP")
    set_singularity_handling(ON)
    print("Speed and compliance setup completed")

    # 갤러리 이미지 처리 및 경로 발굴
    img_path = "/home/shindonghyun/doosan_ws/image/totoro_image.jpg"
    img = cv2.imread(img_path)
    gray_img = rgb2gray(img)
    thresh_val = threshold_otsu(gray_img)
    binary_img = gray_img < thresh_val
    skeleton = skeletonize(binary_img)

# ============================================================
# 이미지 처리(Image Processing & Skeleton Path Extraction)
# ============================================================

    # skel_image 끝점(end_point) 탐색 - 연결된 이웃 pixel의 1개 지점 반환
    def find_endpoints(skel_img):
        endpoints = []
        for y in range(1, skel_img.shape[0] - 1):
            for x in range(1, skel_img.shape[1] - 1):
                if skel_img[y, x] == 1:
                    neighbors = np.sum(skel_img[y-1:y+2, x-1:x+2]) - 1
                    if neighbors == 1:
                        endpoints.append((y, x))
        return endpoints
    
    # BFS 기반 skeleton 경로 추적, 시작점부터 연결된 pixel을 순차적으로 탐색
    def fast_trace_path(skel_img, start):
        visited = np.zeros_like(skel_img, dtype=bool)
        path = []
        q = deque()
        q.append(start)
        visited[start] = True
        while q:
            y, x = q.popleft()
            path.append((y, x))
            for dy in [-1, 0, 1]:
                for dx in [-1, 0, 1]:
                    if dy == 0 and dx == 0:
                        continue
                    ny, nx = y + dy, x + dx
                    if (0 <= ny < skel_img.shape[0]) and (0 <= nx < skel_img.shape[1]):
                        if skel_img[ny, nx] == 1 and not visited[ny, nx]:
                            visited[ny, nx] = True
                            q.append((ny, nx))
        return path

    start_point = (273, 135)
    skeleton_path = fast_trace_path(skeleton, start_point)
    robot_path = [(x, y) for y, x in skeleton_path]

    # 코 이미지 skeleton 경로 추출(이미지 → 이진화 → skeleton → BFS 경로 생성)>
    def nose_path():
        # 1. 이미지 불러오기 및 리사이즈 (210 x 297)
        img_path = "/home/shindonghyun/doosan_ws/image/totoro_nose.jpg"
        img = cv2.imread(img_path)
        # raw = 2408// 10
        # column = 3508 // 10
        # img = cv2.resize(img, (raw, column))  # (width, height)

        # 2. 흑백 변환 및 이진화
        gray_img = rgb2gray(img)
        thresh_val = threshold_otsu(gray_img)
        binary_img = gray_img < thresh_val  # 검정 선 추출을 위해 반전

        # 3. 스켈레톤 중심선 추출
        skeleton = skeletonize(binary_img)

        # 4. 끝점 찾기 함수
        def find_endpoints(skel_img):
            endpoints = []
            for y in range(1, skel_img.shape[0] - 1):
                for x in range(1, skel_img.shape[1] - 1):
                    if skel_img[y, x] == 1:
                        neighbors = np.sum(skel_img[y-1:y+2, x-1:x+2]) - 1
                        if neighbors == 1:
                            endpoints.append((y, x))
            return endpoints
        #642 871
        # 5. BFS 기반 경로 추적
        def fast_trace_path(skel_img, start):
            visited = np.zeros_like(skel_img, dtype=bool)
            path = []
            q = deque()
            q.append(start)
            visited[start] = True
            while q:
                y, x = q.popleft()
                path.append((y, x))
                for dy in [-1, 0, 1]:
                    for dx in [-1, 0, 1]:
                        if dy == 0 and dx == 0:
                            continue
                        ny, nx = y + dy, x + dx
                        if (0 <= ny < skel_img.shape[0]) and (0 <= nx < skel_img.shape[1]):
                            if skel_img[ny, nx] == 1 and not visited[ny, nx]:
                                visited[ny, nx] = True
                                q.append((ny, nx))
            return path

        # 6. 시작점 설정 및 경로 추출
        endpoints = find_endpoints(skeleton)
        start_point = (100, 107)  # 수동 설정 가능: (y, x)
        skeleton_path = fast_trace_path(skeleton, start_point)

        # 7. 시각화 (10개마다 점찍기)
        img_vis = img.copy()
        for idx, (y, x) in enumerate(skeleton_path[::3]):
            cv2.circle(img_vis, (x, y), 2, (0, 0, 255), -1)
            cv2.putText(img_vis, str(idx + 1), (x + 3, y - 3),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.15, (255, 0, 0), 1)

        # plt.figure(figsize=(8, 11))
        # plt.imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
        # plt.title("Skeleton Path (Resized to 210x297)")
        # plt.axis("off")
        # plt.show()

        # 8. 로봇 좌표로 변환
        nose_draw = [(x, y) for y, x in skeleton_path]
        print(len(nose_draw)//5)
        return nose_draw
    
    # 패턴 이미지 여러 개를 skeleton경로로 변환, 각 패턴 별 drawing path 생성
    def path_pattern():
        pat_list = []
        pat_list.append("/home/jin/Downloads/pattern_1.jpg")
        pat_list.append("/home/jin/Downloads/pattern_2.jpg")
        pat_list.append("/home/jin/Downloads/pattern_3.jpg")
        pat_list.append("/home/jin/Downloads/pattern_4.jpg")
        pat_list.append("/home/jin/Downloads/pattern_5.jpg")
        pat_list.append("/home/jin/Downloads/pattern_6.jpg")
        pat_list.append("/home/jin/Downloads/pattern_7.jpg")

        point_list= []
        point_list.append((160, 81))
        point_list.append((142,99))
        point_list.append((172,134))
        point_list.append((176,45))
        point_list.append((181,91))
        point_list.append((188,121))
        point_list.append((204,142))

        list_pattern = []  # 🔄 반복문 밖으로 위치 수정

        for i in range(0,7):    
            img_path = pat_list[i]
            img = cv2.imread(img_path)

            gray_img = rgb2gray(img)
            thresh_val = threshold_otsu(gray_img)
            binary_img = gray_img < thresh_val

            skeleton = skeletonize(binary_img)

            def find_endpoints(skel_img):
                endpoints = []
                for y in range(1, skel_img.shape[0] - 1):
                    for x in range(1, skel_img.shape[1] - 1):
                        if skel_img[y, x] == 1:
                            neighbors = np.sum(skel_img[y-1:y+2, x-1:x+2]) - 1
                            if neighbors == 1:
                                endpoints.append((y, x))
                return endpoints

            def fast_trace_path(skel_img, start):
                visited = np.zeros_like(skel_img, dtype=bool)
                path = []
                q = deque()
                q.append(start)
                visited[start] = True
                while q:
                    y, x = q.popleft()
                    path.append((y, x))
                    for dy in [-1, 0, 1]:
                        for dx in [-1, 0, 1]:
                            if dy == 0 and dx == 0:
                                continue
                            ny, nx = y + dy, x + dx
                            if (0 <= ny < skel_img.shape[0]) and (0 <= nx < skel_img.shape[1]):
                                if skel_img[ny, nx] == 1 and not visited[ny, nx]:
                                    visited[ny, nx] = True
                                    q.append((ny, nx))
                return path

            start_point = point_list[i]
            skeleton_path = fast_trace_path(skeleton, start_point)

            img_vis = img.copy()
            for idx, (y, x) in enumerate(skeleton_path[::2]):
                cv2.circle(img_vis, (x, y), 2, (0, 0, 255), -1)
                cv2.putText(img_vis, str(idx + 1), (x + 3, y - 3),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.15, (255, 0, 0), 1)

            # plt.figure(figsize=(8, 11))
            # plt.imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
            # plt.title("Skeleton Path (Resized to 210x297)")
            # plt.axis("off")
            # plt.show()

            pattern_list = [(x, y) for y, x in skeleton_path]
            print(len(pattern_list)//5)
            list_pattern.append(pattern_list)  # 🔄 여기로 이동

        return list_pattern
    
    # 입 이미지 skeleton 경로 생성 함수
    def path_mouth():
        # 1. 이미지 불러오기 및 리사이즈 (210 x 297)
        img_path = "/home/jin/Downloads/IMG_1451.jpg"
        img = cv2.imread(img_path)
        # raw = 2408// 10
        # column = 3508 // 10
        # img = cv2.resize(img, (raw, column))  # (width, height)

        # 2. 흑백 변환 및 이진화
        gray_img = rgb2gray(img)
        thresh_val = threshold_otsu(gray_img)
        binary_img = gray_img < thresh_val  # 검정 선 추출을 위해 반전

        # 3. 스켈레톤 중심선 추출
        skeleton = skeletonize(binary_img)

        # 4. 끝점 찾기 함수
        def find_endpoints(skel_img):
            endpoints = []
            for y in range(1, skel_img.shape[0] - 1):
                for x in range(1, skel_img.shape[1] - 1):
                    if skel_img[y, x] == 1:
                        neighbors = np.sum(skel_img[y-1:y+2, x-1:x+2]) - 1
                        if neighbors == 1:
                            endpoints.append((y, x))
            return endpoints
        #642 871
        # 5. BFS 기반 경로 추적
        def fast_trace_path(skel_img, start):
            visited = np.zeros_like(skel_img, dtype=bool)
            path = []
            q = deque()
            q.append(start)
            visited[start] = True
            while q:
                y, x = q.popleft()
                path.append((y, x))
                for dy in [-1, 0, 1]:
                    for dx in [-1, 0, 1]:
                        if dy == 0 and dx == 0:
                            continue
                        ny, nx = y + dy, x + dx
                        if (0 <= ny < skel_img.shape[0]) and (0 <= nx < skel_img.shape[1]):
                            if skel_img[ny, nx] == 1 and not visited[ny, nx]:
                                visited[ny, nx] = True
                                q.append((ny, nx))
            return path

        # 6. 시작점 설정 및 경로 추출
        endpoints = find_endpoints(skeleton)
        start_point = (127,111)  # 수동 설정 가능: (y, x)
        skeleton_path = fast_trace_path(skeleton, start_point)

        # 7. 시각화 (10개마다 점찍기)
        img_vis = img.copy()
        for idx, (y, x) in enumerate(skeleton_path[::1]):
            cv2.circle(img_vis, (x, y), 2, (0, 0, 255), -1)
            cv2.putText(img_vis, str(idx + 1), (x + 3, y - 3),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.15, (255, 0, 0), 1)

        # plt.figure(figsize=(8, 11))
        # plt.imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
        # plt.title("Skeleton Path (Resized to 210x297)")
        # plt.axis("off")
        # plt.show()

        # 8. 로봇 좌표로 변환
        mouth_path = [(x, y) for y, x in skeleton_path]
        print(len(mouth_path)//5)
        return mouth_path

# ============================================================
# Rviz 시각화(RViz Marker Visualization)
# ============================================================

    marker_pub = node.create_publisher(Marker, "/skeleton_path_marker", 10)
    marker = Marker()
    marker.header.frame_id = "base_link"
    marker.ns = "skeleton_path"
    marker.id = 0
    marker.type = Marker.LINE_STRIP
    marker.action = Marker.ADD
    marker.scale.x = 0.002
    marker.color.a = 1.0
    marker.color.r = 1.0
    marker.color.g = 0.0
    marker.color.b = 0.0
    marker.pose.orientation.w = 1.0
    marker.points = []

    # RViz 마커 퍼블리셔 설정(skeleton 경로 및 눈, 코 마커를 시각화)
    def marker_timer_callback():
        if marker_timer_callback.idx < len(robot_path):
            x, y = robot_path[marker_timer_callback.idx]
            p = Point()
            p.x = (x + 300.0) / 1000.0
            p.y = (y - 220.0) / 1000.0
            p.z = 0.0
            marker.points.append(p)
            marker.header.stamp = node.get_clock().now().to_msg()
            marker_pub.publish(marker)
            marker_timer_callback.idx += 1

        elif not marker_timer_callback.eyes_drawn:
            # 여기서 눈 마커를 한 번만 추가
            def make_circle_marker(ns, marker_id, color, center, radius_mm, num_points=36):
                m = Marker()
                m.header.frame_id = "base_link"
                m.header.stamp = node.get_clock().now().to_msg()
                m.ns = ns
                m.id = marker_id
                m.type = Marker.LINE_STRIP
                m.action = Marker.ADD
                m.scale.x = 0.002
                m.color.a = 1.0
                m.color.r, m.color.g, m.color.b = color
                m.pose.orientation.w = 1.0

                cx, cy = center
                for i in range(num_points + 1):  # +1 to close the circle
                    angle = 2 * math.pi * i / num_points
                    x = cx + radius_mm * math.cos(angle)
                    y = cy + radius_mm * math.sin(angle)
                    p = Point()
                    p.x = x / 1000.0   # 오프셋 제거
                    p.y = y / 1000.0   # 오프셋 제거
                    p.z = 0.0
                    m.points.append(p)
                return m


            left_eye_marker = make_circle_marker(
                ns="left_eye", marker_id=1, color=(1.0, 0.0, 0.0),
                center=(434.0 + left_eyes_x, 56.0 + left_eyes_y),
                radius_mm=2.0
            )

            right_eye_marker = make_circle_marker(
                ns="right_eye", marker_id=2, color=(1.0, 0.0, 0.0),
                center=(434.0 + right_eyes_x, 56.0 + right_eyes_y),
                radius_mm=2.0
            )


            marker_pub.publish(left_eye_marker)
            marker_pub.publish(right_eye_marker)
           # === ✅ 코 마커 추가 (스켈레톤 기반 경로) ===
            nose_points = nose_path()[::3]  # 간격 조절
            nose_marker = Marker()
            nose_marker.header.frame_id = "base_link"
            nose_marker.header.stamp = node.get_clock().now().to_msg()
            nose_marker.ns = "nose"
            nose_marker.id = 3
            nose_marker.type = Marker.LINE_STRIP
            nose_marker.action = Marker.ADD
            nose_marker.scale.x = 0.002
            nose_marker.color.a = 1.0
            nose_marker.color.r = 1.0
            nose_marker.color.g = 0.0
            nose_marker.color.b = 0.0
            nose_marker.pose.orientation.w = 1.0

            for (x, y) in nose_points:
                p = Point()
                p.x = (x + 300.0) / 1000.0
                p.y = (y - 220.0) / 1000.0
                p.z = 0.0
                nose_marker.points.append(p)

            marker_pub.publish(nose_marker)
            marker_timer_callback.eyes_drawn = True

            marker_timer_callback.eyes_drawn = True  # 두 번 그리지 않게
    
    marker_timer_callback.idx = 0
    marker_timer_callback.eyes_drawn = False

# ============================================================
# 로봇 드롱잉 제어(Robot Drawing Motion Control)
# ============================================================

    # 입 경로를 따라 로봇 드로잉 수행(movex 기반 연속 이동)
    def draw_mouth():
        movel(posx(0,0,100,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

        mouthpath = path_mouth()
        mouth_pattern = mouthpath
        chunks = [[] for _ in range(10)]
        for (x, y) in mouth_pattern[::1]:
            p = posx([float(x)+310.0, float(y)-228.0, 109.0, 118.03, 180.0, 118.08])
            for chunk in chunks:
                if len(chunk) < 80:
                    chunk.append(p)
                    break

        idxss = 0
        while idxss < len(chunks):
            segment = chunks[idxss]
            if segment:
                print(f"[INFO] movesx 실행: chunk {idxss}, pose 수: {len(segment)}")
                movesx(segment, vel=VELOCITY, acc=ACC)
            idxss += 1     
        movel(posx(0,0,100,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

    # 패턴 그리기(토토로 배 패턴 이미지를 순차적으로 드로잉)
    def draw_pattern():
        movel(posx(0,0, 20 ,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

        patternpath = path_pattern()
        for i in range(0, 7): 
            pattern = patternpath[i]
            chunks = [[] for _ in range(1)]
            for (x, y) in pattern[::2]:
                p = posx([float(x)+300.0, float(y)-220.0, 108.5, 118.03, 180.0, 118.08])
                for chunk in chunks:
                    if len(chunk) < 80:
                        chunk.append(p)
                        break

            idx = 0
            while idx < len(chunks):
                segment = chunks[idx]
                if segment:
                    print(f"[INFO] movesx 실행: chunk {idx}, pose 수: {len(segment)}")
                    movesx(segment, vel=VELOCITY, acc=ACC)
                    movel(posx(0,0,100,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                idx += 1
        # for i in range(0,5):
    
    # 코 그리기(토토로 코 skeleton 경로를 로봇으로 드로잉)
    def draw_nose():
        nosepath = nose_path()
        nose_point = posx([434.0 + left_eyes_x + nose_offset_x, 56.0 + left_eyes_y+nose_offset_y, 109.0, 118.03, 180.0, 118.08])
        nose_ready = posx([434.0 + left_eyes_x + nose_offset_x, 56.0 + left_eyes_y+nose_offset_y, 109.0 + 50, 118.03, 180.0, 118.08])
        movel(nose_ready, v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)

        movel(nose_point, v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
        nose_list= [[] for _ in range(10)]
        for (x, y) in nosepath[::3]:
            p = posx([float(x)+310.0, float(y)-205.0, 110.0, 118.03, 180.0, 118.08])
            for nose in nose_list:
                if len(nose) < 80:
                    nose.append(p)
                    break
        index = 0
        while index < len(nose_list):
            segment = nose_list[index]
            if segment:
                print(f"[INFO] movesx 실행: chunk {index}, pose 수: {len(segment)}")
                movesx(segment, vel=VELOCITY, acc=ACC)
            index += 1
        movel(posx(0,0, 100 ,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

    # 눈 그리기(힘제어(compliance control과 force control)기반 토토로 눈 드로잉 수행)
    def draw_eyes():
        #눈 그리기 
        time.sleep(1.0)

        # 왼쪽 눈
        left_eyes_point = posx([434.0 + left_eyes_x, 56.0 + left_eyes_y, 109.0, 118.03, 180.0, 118.08])
        left_eyes_ready = posx([434.0 + left_eyes_x, 56.0 + left_eyes_y, 109.0 + 50, 118.03, 180.0, 118.08])

        movel(left_eyes_ready, v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)

        movel(left_eyes_point, v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)

        # move_spiral(rev=3.0, rmax=10.0, lmax=30.0, vel=10, acc=20, time=10.0, axis=DR_AXIS_Z, ref=DR_TOOL)
        task_compliance_ctrl(stx=[500, 500, 500, 100, 100, 100])
        
        time.sleep(0.1)
        # set_stiffnessx([3000.0]*3 + [200.0]*3, time=0.0)
        # wait(0.5)
        set_desired_force(fd=[0, 0, -8, 0, 0, 0], dir=[0, 0, 1, 0, 0, 0], mod=DR_FC_MOD_REL)  
        time.sleep(3.0)
        release_force()
        time.sleep(0.1)
        release_compliance_ctrl()
        time.sleep(0.1)     
        # # 오른쪽 눈
        time.sleep(1.0)
        movel(posx(0,0, 100 ,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

        right_eyes_point = posx([434.0 + right_eyes_x, 56.0 + right_eyes_y, 109.0, 118.03, 180.0, 118.08])
        right_eyes_ready = posx([434.0 + right_eyes_x, 56.0 + right_eyes_y, 109.0 + 50, 118.03, 180.0, 118.08])

        movel(right_eyes_ready, v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)

        movel(right_eyes_point, v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
        task_compliance_ctrl(stx=[500, 500, 500, 100, 100, 100])
        time.sleep(0.1)
        set_desired_force(fd=[0, 0, -10, 0, 0, 0], dir=[0, 0, 1, 0, 0, 0], mod=DR_FC_MOD_REL)  
        time.sleep(3.0)
        release_force()
        time.sleep(0.1)
        release_compliance_ctrl()
        time.sleep(0.1)  
        # move_spiral(rev=9.5, rmax=20.0, lmax=50.0, vel=20, acc=50, time=20.0, axis=DR_AXIS_Z, ref=DR_TOOL)
        movel(posx(0,0, 100 ,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

        print("Pick-and-place sequence completed.")

    # 일반 펜을 내려놓고 붓펜으로 교체
    def brush_pen():
        get_pen = posx([223.88, -182.77, 117.37, 69.39, -179.99, 90.71])
        get_pen_up = posx([223.88, -182.77, 211.37, 69.39, -179.99, 90.71])
        movel(get_pen_up, v=20, a=20, vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
        movel(get_pen, v=20, a=20, vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
        release()
        time.sleep(1.0)
        movel(get_pen_up, v=20, a=20, vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
        get_brushpen = posx([200.12, -58.35, 130.0, 152.13, -179.99, 151.76])
        ready_brushpen = posx([200.12, -58.35, 213.87, 152.13, -179.99, 151.76])
        movel(ready_brushpen, v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)

        movel(get_brushpen, v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
        grip()
        time.sleep(3.0)
        movel(ready_brushpen, v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)

# ============================================================
# 힘제어(Force Control & Human Interaction)
# ============================================================

    # 사용자의 force 입력(Nudge)을 감지 후 드로잉 수행
    def wait_for_nudge_then_draw(threshold=20.0):
            print("Z축 방향으로 최소 20.0N 이상 누르면 다음 동작을 수행합니다.")
            try:
                while rclpy.ok():
                    while not check_force_condition(axis=DR_AXIS_Z, min=threshold, ref=DR_TOOL):
                        print("Nudge 감지됨! 후속 동작 수행 중")
                        brush_pen()
                        draw_eyes()
                        # draw_nose()
                        break
                    time.sleep(0.1)                        
                    
                print("경로 실행 중...")
                time.sleep(3.0)
                print("경로 실행 완료")        
            except Exception as e:
                print(f"예외 발생: {e}")   
    # # 펜 pick 동작 수행
    def pen_pick_and_place():
        # 펜잡기
        get_pen = posx([223.88, -182.77, 118.00, 69.39, -179.99, 90.71])
        get_pen_up = posx([223.88, -182.77, 211.37, 69.39, -179.99, 90.71])
        release()
        movel(get_pen_up, v=20, a=20, vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
        movel(get_pen, v=20, a=20, vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
        grip()
        time.sleep(3.0)
        movel(get_pen_up, v=20, a=20, vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
    # 그리퍼 open
    def release():
        set_digital_output(2, ON)
        set_digital_output(1, OFF)
        wait(0.5)
    # 그리퍼 close
    def grip():
        set_digital_output(2, OFF)
        set_digital_output(1, ON)
        wait(0.5)
    pen_pick_and_place()

    # 로봇 동작 시작
    JReady = [0, -20, 110, 0, 90, 0]
    movej(JReady, v=20, a=20)
    Ready = posx([434.0, 56.0, 109.0, 118.03, 180.0, 118.08])
    movel(Ready, v=20, a=20, vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)

    # 경로 이동 분할
    chunks = [[] for _ in range(10)]
    for (x, y) in robot_path[5::5]:
        p = posx([float(x)+300.0, float(y)-220.0, 109.0, 118.03, 180.0, 118.08])
        for chunk in chunks:
            if len(chunk) < 80:
                chunk.append(p)
                break
            
    node.create_timer(0.11, marker_timer_callback)

    idx = 0
    while idx < len(chunks):
        segment = chunks[idx]
        if segment:
            print(f"[INFO] movesx 실행: chunk {idx}, pose 수: {len(segment)}")
            movesx(segment, vel=VELOCITY, acc=ACC)
        idx += 1
    # # 넛지 
    draw_mouth()
    draw_pattern()
    draw_nose()
    wait_for_nudge_then_draw()

    # rclpy.spin(node)
    rclpy.spin_once(node)
    rclpy.shutdown()

if __name__ == '__main__':
    main()


### GUI 코드

In [ ]:
import sys
import cv2 # OpenCV, 영상 재생 및 프레임 처리
import rclpy # ROS2 노드 기능
from rclpy.node import Node
# GUI 구축 (영상 표시, 슬라이더 포함)
from PyQt5.QtWidgets import ( 
    QApplication, QLabel, QWidget, QHBoxLayout, QVBoxLayout,
    QSlider, QDesktopWidget, QMainWindow, QFrame
)
from PyQt5.QtGui import QImage, QPixmap
from PyQt5.QtCore import QTimer, Qt


# ------------------ 영상만 보여주는 창 ------------------ #
class VideoWindow(QWidget):
    def __init__(self, left_path, right_path):
        super().__init__()
        self.setWindowTitle("rviz - 토토로  /  실제 - 토토로")  # 창 제목 설정

        # ▶ 왼쪽과 오른쪽 영상 파일 열기
        self.left_cap = cv2.VideoCapture(left_path)   # 왼쪽 영상 스트림
        self.right_cap = cv2.VideoCapture(right_path) # 오른쪽 영상 스트림

        # ▶ 왼쪽 영상의 원본 해상도 정보 가져오기
        self.original_width = int(self.left_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.original_height = int(self.left_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # ▶ 영상 크기 비율 초기값 설정 (0.5 = 50% 크기로 시작)
        self.scale_ratio = 0.5

        # ▶ QLabel 생성: 각 영상 프레임을 표시할 위젯
        self.left_label = QLabel()
        self.right_label = QLabel()

        # ▶ QLabel에 영상을 꽉 차게 표시할 수 있도록 설정
        self.left_label.setScaledContents(True)
        self.right_label.setScaledContents(True)

        # ▶ 수평 레이아웃에 QLabel 두 개 배치 (좌우로 영상 나란히)
        layout = QHBoxLayout()
        layout.addWidget(self.left_label)
        layout.addWidget(self.right_label)
        self.setLayout(layout)

        # ▶ 타이머 생성: 일정 간격마다 프레임을 업데이트
        self.timer = QTimer()
        self.timer.timeout.connect(self.update_frames)  # 프레임 갱신 함수 연결
        self.timer.start(30)  # 30ms 간격으로 프레임 갱신 (~33fps)

        # ▶ 초기 창 크기 설정 (화면 크기 기반, 비율 유지)
        self.init_size()

    def init_size(self):
        # ▶ 현재 화면 해상도 가져오기
        screen = QDesktopWidget().screenGeometry()
        screen_width = screen.width()
        screen_height = screen.height()

        # ▶ 영상의 가로세로 비율 계산
        aspect_ratio = self.original_width / self.original_height

        # ▶ 창의 최대 너비/높이 = 화면의 50% 수준
        max_width = int(screen_width * 0.5)
        max_height = int(screen_height * 0.5)

        # ▶ 가로 기준으로 창 크기 계산 (비율 유지)
        width = max_width
        height = int(width / aspect_ratio)

        # ▶ 계산된 높이가 최대 높이를 넘는다면 높이 기준으로 다시 계산
        if height > max_height:
            height = max_height
            width = int(height * aspect_ratio)

        # ▶ 최종 크기로 창 크기 조정
        self.resize(width, height)

    def set_scale(self, ratio):
        # ▶ 외부 슬라이더로부터 영상 스케일 값을 전달받아 저장
        self.scale_ratio = ratio

    def resize_window_by_ratio(self, ratio):
        # ▶ 외부 슬라이더로부터 창 크기 배율을 전달받아 창 전체 크기 조정
        screen = QDesktopWidget().screenGeometry()
        w = int(screen.width() * ratio)
        h = int(screen.height() * ratio)
        self.resize(w, h)

    def update_frames(self):
        # ▶ 왼쪽/오른쪽 영상에서 각각 프레임 읽기
        ret_left, frame_left = self.left_cap.read()
        ret_right, frame_right = self.right_cap.read()

        # ▶ 프레임을 정상적으로 읽은 경우, 스케일 조정 후 표시
        if ret_left:
            resized = self.scale_frame(frame_left)
            self.left_label.setPixmap(self.convert_frame(resized))

        if ret_right:
            resized = self.scale_frame(frame_right)
            self.right_label.setPixmap(self.convert_frame(resized))

    def scale_frame(self, frame):
        # ▶ 현재 scale_ratio에 따라 프레임을 리사이즈 (비율 유지)
        w = int(self.original_width * self.scale_ratio)
        h = int(self.original_height * self.scale_ratio)
        return cv2.resize(frame, (w, h), interpolation=cv2.INTER_AREA)

    def convert_frame(self, frame):
        # ▶ OpenCV BGR → RGB 변환 → QImage → QPixmap 변환
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w, ch = rgb.shape
        return QPixmap.fromImage(QImage(rgb.data, w, h, ch * w, QImage.Format_RGB888))



# ------------------ 슬라이더 전용 GUI 창 (영상 없음) ------------------ #
class ControlWindow(QWidget):
    def __init__(self, video_window):
        super().__init__()
        self.setWindowTitle("영상 제어 패널")
        self.video_window = video_window  # 조절할 VideoWindow 인스턴스 연결

        # ▶ 해상도 조절 슬라이더 (10% ~ 200%)
        self.scale_slider = QSlider(Qt.Horizontal)
        self.scale_slider.setMinimum(10)
        self.scale_slider.setMaximum(100)
        self.scale_slider.setValue(50)  # 초기값 100%
        self.scale_slider.valueChanged.connect(self.update_scale)

        # ▶ 창 크기 조절 슬라이더 (50% ~ 150%)
        self.window_slider = QSlider(Qt.Horizontal)
        self.window_slider.setMinimum(10)
        self.window_slider.setMaximum(100)
        self.window_slider.setValue(50)  # 초기값 100%
        self.window_slider.valueChanged.connect(self.update_window_size)

        # 슬라이더 레이아웃 구성
        layout = QVBoxLayout()
        layout.addWidget(QLabel("해상도 조절 (배율%)"))
        layout.addWidget(self.scale_slider)
        layout.addSpacing(20)
        layout.addWidget(QLabel("창 크기 조절 (화면 비율%)"))
        layout.addWidget(self.window_slider)

        self.setLayout(layout)

        # 슬라이더 패널의 창 크기는 고정
        self.setFixedSize(400, 200)


    def update_scale(self, value):
        # 해상도 배율 변경 시 VideoWindow에 전달
        self.video_window.set_scale(value / 100.0)


    def update_window_size(self, value):
        # GUI 창 크기 비율 변경 시 VideoWindow에 전달
        self.video_window.resize_window_by_ratio(value / 100.0)



# ------------------ 메인: 영상창 + 슬라이더창 동시에 실행 ------------------ #
def main(args=None):
    rclpy.init(args=args)
    app = QApplication(sys.argv)

    # 1. 영상 표시용 GUI 창 생성 및 실행
    video_window = VideoWindow(
        "/home/shindonghyun/doosan_ws/image/rviz_draw.mp4",
        "/home/shindonghyun/doosan_ws/image/real_draw.mp4"
    )
    video_window.show()

    # 2. 슬라이더만 있는 제어 GUI 창 생성 및 실행 (video_window를 조절)
    control_window = ControlWindow(video_window)
    control_window.show()

    # Qt 이벤트 루프 실행
    sys.exit(app.exec_())



if __name__ == "__main__":
    main()




## 2. 유성기어 조립 공정 자동화 시스템

### 👉 역할
 로봇 팔 이동 좌표 추출 및 동선 최적화, 컴플라이언스/힘 제어 알고리즘 구현
### 👉 개발 환경
하드웨어 - 협동로봇(Doosan m0609), PC, 유성기어 모듈

소프트웨어 - Ubuntu 22.04(ROS2 Humble), Python, Doosan API
### 👉 프로젝트 개요
협동로봇(Doosan M0609)으로 유성기어 조립과 3×3 팔레타이징 공정 자동화 구현.
### 👉 프로젝트 목표
로봇 팔의 이동 경로 최적화로 사이클 타임 단축.

협동로봇 안전 범위 내 접촉 테스트로 작업 안정성 강화.

힘 제어, 컴플라이언스 제어 기반의 공정 정밀도와 안정성 향상.

### 👉 주요성과 및 문제해결
직접 교시로 이동 좌표를 추출 중에 제어기 과전류 오류로 인한 충돌 사고 위험이 발생해 공정 특성 고려 상대 이동 기반 제어로 전환하고 안전거리 기준을 재정립해 해결.

Force Compliance 기반 유동적 제어를 적용 z축 미세 보정을 통해 조립 공정 정밀도 개선.

radius 블렌딩 적용 로봇 팔 정지/재가속 시간 감소와 그리퍼 비동기 제어를 통해 공정 사이클 타임을 단축.


### Task 환경 구성
<img src="images/gear_task.png" width="600">


### 유성기어란?
<img src="images/3gear.png" width="600">

구조: 중앙의 선기어(Sun), 이를 둘러싼 3개 이상의 유성기어(Planet), 그리고 가장 바깥쪽의 링기어(Ring)로 구성됩니다.

장점: 소형 크기로도 큰 동력 전달이 가능하며, 높은 토크를 처리할 수 있습니다.

활용: 자동 변속기, 산업용 정밀 감속기, 로봇 등에서 감속비(3:1 ~ 100:1 이상)를 구현하는 데 필수적인 요소입니다.

### 기어 조립 제어 코드

In [ ]:
import rclpy
import DR_init
import time
import numpy as np
from collections import deque

ROBOT_ID = "dsr01"
ROBOT_MODEL = "m0609"
VELOCITY, ACC = 100, 100

DR_init.__dsr__id = ROBOT_ID
DR_init.__dsr__model = ROBOT_MODEL

OFF, ON = 0, 1

def main(args=None):
    rclpy.init(args=args)
    node = rclpy.create_node("doosan_pick_and_place", namespace=ROBOT_ID)
    DR_init.__dsr__node = node

    try:
        from DSR_ROBOT2 import (
            set_tool, set_tcp, movej, movel, wait,
            set_digital_output, set_singularity_handling,
            set_desired_force, release_force, release_compliance_ctrl,
            check_force_condition, move_periodic, check_position_condition, task_compliance_ctrl,
            DR_BASE, DR_TOOL, DR_AXIS_Z,
            DR_MV_MOD_REL, DR_FC_MOD_REL,DR_MV_MOD_ABS,
            set_stiffnessx,
            get_current_posx,amove_periodic
        )
        from DR_common2 import posj, posx

    except ImportError as e:
        print(f"Error importing Doosan API modules: {e}")
        return

    # 초기 세팅
    set_tool("Tool Weight_test")
    set_tcp("test_TCP")
    set_singularity_handling(ON)
    print("Speed and compliance setup completed")
 

    def release():
        set_digital_output(2, ON)
        set_digital_output(1, OFF)
        wait(0.5)

    def close():
        set_digital_output(2, OFF)
        set_digital_output(1, ON)
        wait(3.0)

    # def set_compliance():
    #     wait(0.1)
    #     set_desired_force([0.0, 0.0, -10.0, 0, 0, 0], [0, 0, 1, 0, 0, 0], time=0.0, mod=DR_FC_MOD_ABS)

    def center_gear():
        movel(posx(0.0, 0.0, -100.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
        close()
        movel(posx(0.0, 0.0, 150.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=68.18, ref=DR_BASE, mod=DR_MV_MOD_REL)
        movel(posx(300.0, 0.0, 0.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=139.61, ref=DR_BASE, mod=DR_MV_MOD_REL)
        movel(posx(0.0, 0.0, -115.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
        # wait(3.0)
        task_compliance_ctrl(stx=[500, 500, 500, 100, 100, 100])
        # set_stiffnessx([3000.0]*3 + [200.0]*3, time=0.0)
        wait(0.5)
        set_desired_force(fd=[0, 0, -10, 0, 0, 0], dir=[0, 0, 1, 0, 0, 0], mod=DR_FC_MOD_REL)

        wait(1.0)
        detecting()

    def pick_and_place():
        movel(posx(0.0, 0.0, -100.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
        close()
        movel(posx(0.0, 0.0, 100.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=100.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
        movel(posx(300.0, 0.0, 0.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=100.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
        movel(posx(0.0, 0.0, -100.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=30.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
        release()
        wait(0.5)
        movel(posx(0.0, 0.0, 100.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=30.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

    def detecting():
        while rclpy.ok():
            while not check_force_condition(axis=DR_AXIS_Z, min=10, ref=DR_TOOL):
                print("힘 체크 완료")
                amove_periodic(
                    amp=[0, 0, 0, 0, 0, 20],
                    period=[0, 0, 0, 0, 0, 1.5],
                    atime=0.0, repeat=3, ref=DR_TOOL
                )
                time.sleep(0.1)
                print(get_current_posx())
                value_z = get_current_posx(ref=DR_BASE)[0][2]
                time.sleep(0.1)
                
                print(f"{value_z}")
                time.sleep(0.1)
                if value_z < 65:
                    print("포지션 체크")
                    # print(f"{value_z}")

                    release()
                    release_force(time=0.0)
                    movel(posx(0.0, 0.0, 100.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                    release_compliance_ctrl()
                    break


    # 초기 자세로 이동
    # movel(posx([0, 0, 90, 0, 180, 0]), vel=VELOCITY, acc=ACC, radius=30.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
    JReady = [0, -20, 110, 0, 90, 0]
    movej(JReady, v=20, a=20)
    Ready = posx([567.68, 112.22, 100, 118.03, 180, 118.08])
    movel(Ready, v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)


    # 위치 설정(각기어 중심점 위치)
    place1 = posx([214.47, -5.91, 155.0, 116.43, -177.79, 119.23])
    place2 = posx([302.56, 53.13, 155.0, 123.30, 178.09, 123.12])
    place3 = posx([309.74, -48.01, 155.0, 132.11, 178.60, 149.86])
    place4 = posx([274.77, 1.97, 155.0, 93.90, 179.10, 90.43])

    release()
    movel(place1, vel=VELOCITY, acc=ACC)
    pick_and_place()

    movel(place2, vel=VELOCITY, acc=ACC)
    pick_and_place()

    movel(place3, vel=VELOCITY, acc=ACC)
    pick_and_place()
    movel(posx(0.0, 0.0, 100.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

    movel(place4, vel=VELOCITY, acc=ACC)
    release()
    center_gear()

    print("gear_assemble sequence completed.")
    rclpy.shutdown()

if __name__ == '__main__':
    main()

## 3. 팔레타이징 공정 자동화 시스템

### 👉 역할
로봇 팔 이동 좌표 추출 및 동선 최적화, Stack활용 정렬 알고리즘 구현
### 👉 개발 환경
하드웨어 - 협동로봇(Doosan m0609), PC, 3X3 팔레트 모듈

소프트웨어 - Ubuntu 22.04(ROS2 Humble), Python, Doosan API
### 👉 프로젝트 개요
협동로봇(Doosan M0609)으로 유성기어 조립과 3×3 팔레타이징 공정 자동화 구현.
### 👉 프로젝트 목표
로봇 팔의 이동 경로 최적화로 사이클 타임 단축.
협동로봇 안전 범위 내 접촉 테스트로 작업 안정성 강화.
힘 제어, 컴플라이언스 제어 기반의 공정 정밀도와 안정성 향상.
### 👉 주요성과 및 문제해결
직접 교시로 이동 좌표를 추출 중에 제어기 과전류 오류로 인한 충돌 사고 위험이 발생해 공정 특성 고려 상대 이동 기반 제어로 전환하고 안전거리 기준을 재정립해 해결.

Force Compliance 기반 유동적 제어를 적용 z축 미세 보정을 통해 조립 공정 정밀도 개선.

radius 블렌딩 적용 로봇 팔 정지/재가속 시간 감소와 그리퍼 비동기 제어를 통해 공정 사이클 타임을 단축.

### 팔레타이징 공정 시나리오
<img src="images/palletazing.png" width="600">


### 팔레타이징 공정 제어 코드

In [ ]:
import rclpy
import DR_init
import time
from collections import deque
ROBOT_ID = "dsr01"
ROBOT_MODEL = "m0609"
VELOCITY, ACC = 100, 100

DR_init.__dsr__id = ROBOT_ID
DR_init.__dsr__model = ROBOT_MODEL

OFF, ON = 0, 1

def main(args=None):
    rclpy.init(args=args)
    node = rclpy.create_node("doosan_pick_and_place", namespace=ROBOT_ID)
    DR_init.__dsr__node = node

    try:
        from DSR_ROBOT2 import (
            set_tool, set_tcp, movel, wait,
            set_digital_output, set_singularity_handling,
            set_desired_force, release_force, release_compliance_ctrl,
            check_force_condition, move_periodic, check_position_condition, task_compliance_ctrl,
            DR_BASE, DR_TOOL, DR_AXIS_Z,
            DR_MV_MOD_REL, DR_MV_MOD_ABS,
            DR_FC_MOD_REL,
            get_current_posx
        )
        from DR_common2 import posj, posx

    except ImportError as e:
        print(f"Error importing Doosan API modules: {e}")
        return
                
# 변수 모음
    direction = 1
    row = 3
    col = 3
    stack = 1
    thickness = 0
    point_offset = [0,0,0]
    column = 3
    v = 200
    a = 50
    
    #Total count
    off_set_x = 0
    off_set_y = 0
    total_count = row * column * stack

    j = 0
    k = 3
    z = 6
    value_z = 0
    # 초기 세팅
    set_tool("Tool Weight_test")
    set_tcp("test_TCP")
    set_singularity_handling(ON)
    print("Speed and compliance setup completed")
    pos1 = posx(247.98, 99.0, 38.26, 0, 180, 0)
    basket = []
    basket.append(posx(248.00, -51, 140.00, 0, 180, 0))
    basket.append(posx(298.00, -51, 140.00, 0, 180, 0))
    basket.append(posx(348.00, -51, 140.00, 0, 180, 0))
    basket.append(posx(248.00, -101, 140.00, 0, 180, 0))
    basket.append(posx(298.00, -101, 140.00, 0, 180, 0))
    basket.append(posx(348.00, -101, 140.00, 0, 180, 0))
    basket.append(posx(248.00, -151, 140.00, 0, 180, 0))
    basket.append(posx(298.00, -151, 140.00, 0, 180, 0))
    basket.append(posx(348.00, -151, 140.00, 0, 180, 0))

    def grip():
        #  robot
        # set robot -> gripper
        # get gripper->robot check
        set_digital_output(1,1)
        set_digital_output(2,0)    
        wait(0.3)
        #wati(0.1)

    def release():
        set_digital_output(1,0)
        set_digital_output(2,1)    
        wait(0.3)

    def detecting():
        value_z = 0.0  # ← 반드시 초기화!
        while rclpy.ok():
            while not check_force_condition(axis=DR_AXIS_Z, min=10, ref=DR_TOOL):
                print("힘 체크 완료")
                release_force(time=0.0)
                release_compliance_ctrl()
                time.sleep(0.1)
                value_z = get_current_posx(ref=DR_BASE)[0][2]
                time.sleep(0.1)
                print(f"{value_z}")
                return value_z
        

    for i in range(0, total_count):
        # 제일 긴거 81,
        if i < 3 : 
            pos1[0] = pos1[0] + off_set_x
            
            movel(posx(247.98+off_set_x, 99.0, 38.26 + 80, 0, 180, 0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
            grip()

            task_compliance_ctrl(stx=[500, 500, 500, 100, 100, 100])
            # set_stiffnessx([3000.0]*3 + [200.0]*3, time=0.0)
            wait(0.5)
            set_desired_force(fd=[0, 0, -10, 0, 0, 0], dir=[0, 0, 1, 0, 0, 0], mod=DR_FC_MOD_REL)
            value_z = detecting()
            movel(posx(247.98+off_set_x, 99.0, 100, 0, 180, 0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
            release()
            movel(posx(0,0,-60,0,0,0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            grip()
            movel(posx(0,0, 100,0,0,0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

            if value_z > 80:
                movel(basket[j], v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
                wait(0.5)
                movel(posx(0,0,-90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                release()
                wait(3.0)
                movel(posx(0,0,90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

                j = j +1
                off_set_x = off_set_x + 50
            elif value_z > 70:
                movel(basket[k], v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
                movel(posx(0,0,-90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                release()
                wait(3.0)
                movel(posx(0,0,90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

                k = k + 1
                off_set_x = off_set_x + 50
            else:
                movel(basket[z], v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
                movel(posx(0,0,-90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                release()
                wait(3.0)
                movel(posx(0,0,90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                z = z + 1
                off_set_x = off_set_x + 50
        elif i < 6:
            if i == 3:
                off_set_x = 0
                off_set_y = 50
            pos1[0] = pos1[0] + off_set_x
            pos1[1] = pos1[1] + off_set_y
            
            movel(posx(247.98+off_set_x, 99.0 - off_set_y, 38.26 + 100, 0, 180, 0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
            grip()

            task_compliance_ctrl(stx=[500, 500, 500, 100, 100, 100])
            # set_stiffnessx([3000.0]*3 + [200.0]*3, time=0.0)
            wait(0.5)
            set_desired_force(fd=[0, 0, -10, 0, 0, 0], dir=[0, 0, 1, 0, 0, 0], mod=DR_FC_MOD_REL)
            value_z = detecting()
            movel(posx(247.98+off_set_x, 99.0-off_set_y, 100, 0, 180, 0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
            release()
            movel(posx(0,0,-60,0,0,0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            grip()
            movel(posx(0,0, 100,0,0,0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)  
            if value_z > 80:
                movel(basket[j], v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
                wait(0.5)
                movel(posx(0,0,-90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                release()
                wait(3.0)
                movel(posx(0,0,90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

                j = j +1
                off_set_x = off_set_x + 50
            elif value_z > 70:
                movel(basket[k], v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
                movel(posx(0,0,-90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                release()
                wait(3.0)
                movel(posx(0,0,90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

                k = k + 1
                off_set_x = off_set_x + 50
            else:
                movel(basket[z], v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
                movel(posx(0,0,-90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                release()
                wait(3.0)
                movel(posx(0,0,90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                z = z + 1
                off_set_x = off_set_x + 50     
        elif i < 9:
            if i == 6:
                off_set_x = 0
                off_set_y = 100
            pos1[0] = pos1[0] + off_set_x
            pos1[1] = pos1[1] + off_set_y
            
            movel(posx(247.98+off_set_x, 99.0 - off_set_y, 38.26 + 100, 0, 180, 0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
            grip()

            task_compliance_ctrl(stx=[500, 500, 500, 100, 100, 100])
            # set_stiffnessx([3000.0]*3 + [200.0]*3, time=0.0)
            wait(0.5)
            set_desired_force(fd=[0, 0, -10, 0, 0, 0], dir=[0, 0, 1, 0, 0, 0], mod=DR_FC_MOD_REL)
            value_z = detecting()
            movel(posx(247.98+off_set_x, 99.0-off_set_y, 100, 0, 180, 0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
            release()
            movel(posx(0,0,-60,0,0,0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            grip()
            movel(posx(0,0, 100,0,0,0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)  
            if value_z > 80:
                movel(basket[j], v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
                wait(0.5)
                movel(posx(0,0,-90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                release()
                wait(3.0)
                movel(posx(0,0,90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

                j = j +1
                off_set_x = off_set_x + 50
            elif value_z > 70:
                movel(basket[k], v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
                movel(posx(0,0,-90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                release()
                wait(3.0)
                movel(posx(0,0,90,0,0,0), v=10, a=10 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

                k = k + 1
                off_set_x = off_set_x + 50
            else:
                movel(basket[z], v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_ABS)
                movel(posx(0,0,-90,0,0,0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                release()
                wait(3.0)
                movel(posx(0,0,90,0,0,0), v=20, a=20 ,vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                z = z + 1
                off_set_x = off_set_x + 50 
    print("Pick-and-place sequence completed.")
    rclpy.shutdown()

if __name__ == '__main__':
    main()
